# 1 - Configurações Iniciais



## 1.1 - Instalações

In [ ]:
!pip install minigrid

## 1.2 - Importações

In [ ]:
import time
import os
import sys
import re
import io
from google.colab import userdata

import gymnasium as gym
import minigrid

from google import genai

## 1.3 - Configuração do LLM

In [ ]:
API_KEY = userdata.get('GOOGLE_API_KEY')

GEMINI_CLIENT = genai.Client(api_key=API_KEY)

#GEMINI_MODEL_ID = "gemini-2.5-flash"
#GEMINI_MODEL_ID = "gemini-3.1-flash-lite-preview"
GEMINI_MODEL_ID = "gemini-3-flash-preview"

# 2 - Códigos Auxiliares

- Para representar o mapa de forma textual.
- Para gerar saídas com o modelo.

In [ ]:
MAP_CELLS = {'wall': '#', 'floor': '.', 'goal': 'G', 'key': 'K', 'lava': 'L', 'open_door': '_', 'locked_dor': 'X', 'unlocked_closed_door': 'D'}
PLAYER_DIRECTIONS = ['>', 'v', '<', '^']

class MiniGridTextWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.map_cells_repr = MAP_CELLS
        self.directions_repr = PLAYER_DIRECTIONS
        self.unknown = "?"

    def observation(self, obs):
        env = self.unwrapped
        agent_pos = env.agent_pos
        current_room = None
        if hasattr(env, 'rooms'):
            for room in env.rooms:
                top_x, top_y = room.top
                w, h = room.size
                if (top_x <= agent_pos[0] < top_x + w and top_y <= agent_pos[1] < top_y + h):
                    current_room = room
                    break

        x_s, y_s = (current_room.top, current_room.size) if current_room else ((0,0), (env.grid.width, env.grid.height))

        # Grid logic for the loop execution:
        grid_str = ""
        last_line = y_s[1] - 1
        for y in range(y_s[1]):
            line = ""
            for x in range(x_s[0], x_s[0] + y_s[0]):
                real_x, real_y = x, y + x_s[1]
                if [real_x, real_y] == list(agent_pos):
                    line += self.directions_repr[env.agent_dir]
                else:
                    tile = env.grid.get(real_x, real_y)
                    if tile is None:
                        line += self.map_cells_repr['floor']
                    elif tile.type == 'door':
                        line += (MAP_CELLS['open_door'] if tile.is_open else (MAP_CELLS['locked_dor'] if tile.is_locked else MAP_CELLS['unlocked_closed_door']))
                    else:
                        line += self.map_cells_repr.get(tile.type, self.unknown)
            grid_str += line + ("\n" if y != last_line else "")

        return { "mission": env.mission , "text_grid": grid_str }

In [ ]:
def andar_para_frente():
    obs, r, termi, trunc, _ = env.step(2)
    return obs, r, (termi or trunc)

def girar_para_esquerda():
    obs, r, termi, trunc, _ = env.step(0)
    return obs, r, (termi or trunc)

def girar_para_direita():
    obs, r, termi, trunc, _ = env.step(1)
    return obs, r, (termi or trunc)

def abrir_porta():
    obs, r, termi, trunc, _ = env.step(5)
    return obs, r, (termi or trunc)

# 3 - Testando o Ambiente

Estes ambientes são tarefas simples, da biblioteca **MiniGrid** - https://minigrid.farama.org/environments/minigrid/CrossingEnv/.

Todas envolvem mover um "personagem" por um ambiente definido como malha quadricular.

Veja as definições das ações no link.

In [ ]:
#env = gym.make("MiniGrid-MultiRoom-N2-S4-v0")
env = gym.make("MiniGrid-LavaCrossingS9N3-v0")

In [ ]:
env = MiniGridTextWrapper(env)

obs, _ = env.reset()

print("Observação inicial:")
print("-- missão --")
print(obs['mission'])
print("-- estado do ambiente --")
print(obs['text_grid'])

In [ ]:
# loop aplicando sequência de 5 ações aleatórias
for _ in range(3):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

    print("-- ação --")
    print(action)

    print("-- estado --")
    print(obs['text_grid'])

    #print("-- recompensa --")
    #print(reward)

    if terminated or truncated:
        break

# 4 - Agente CodeAct

Agora, vamos criar um agente CodeAct, baseado em LLM, para mover esse personagem.

## 4.1 - Prompts

In [ ]:
SYSTEM_PROMPT_CODEACT = """
Você é um agente CodeAct controlando um ambiente Gymnasium MiniGrid via *funções de ação* Python.

# REGRAS
1. Responda APENAS com código entre <execute> e </execute>. Qualquer outro formato será rejeitado.
2. Variáveis disponíveis no ambiente: `obs`, `encerrado`, `recompensa` (resultado da última ação).
3. Toda ação deve ser chamada como: `obs, recompensa, encerrado = <função_de_ação>()`
4. Por turno: 1–5 chamadas de ação. Use `print()` para registrar o que precisar decidir no próximo turno.
5. Não precisa resolver em um turno. Comece simples — uma ação por turno inicialmente. Verifique `encerrado` entre ações.

# OBSERVAÇÃO (`obs`)
Dicionário com:
- `'mission'`: objetivo atual
- `'text_grid'`: mapa em grade de texto

Células: `#`=parede, `.`=chão, `D`=porta fechada, `_`=porta aberta, `G`=objetivo, `L`=lava
Agente: `^` `v` `<` `>` (posição + direção)

# AÇÕES (sem parâmetros)
- `andar_para_frente()` — move um quadrado à frente (se for `.`)
- `girar_para_esquerda()` — gira 90° anti-horário
- `girar_para_direita()` — gira 90° horário
- `abrir_porta()` — abre porta `D` à frente

Mover para fora da sala leva a outra sala.
"""

In [ ]:
INITIAL_OBSERVATION_TEMPLATE = """
# OBSERVAÇÃO INICIAL

## mission
{MISSION}

## text_grid
{GRID}

Agora, *escreva código* para controlar o agente.
"""

In [ ]:
NEXT_OBSERVATION_TEMPLATE = """
NOVA OBSERVAÇÃO (STDOUT):
{STDOUT_LOG}
---
Agora, *escreva código* para controlar o agente.
"""

## 4.2 - Ambiente de Execução de Código

Esta função executa código Python informado no parâmetro `code_str`.

In [ ]:
def run_codeact_snippet(code_str, exec_globals, exec_locals):
    stdout_buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = stdout_buffer

    try:
        exec(code_str, exec_globals, exec_locals)
    finally:
        sys.stdout = old_stdout

    return stdout_buffer.getvalue()


In [ ]:
#my_var = 0
#context = {"my_var": my_var}
#exec("for i in range(2): my_var = 1", context)
#print(my_var)             # não muda
#print(context['my_var'])  # muda


## 4.3 - Execução

In [ ]:
def print_observation(turn, code, stdout_log, recompensa, encerrado):
    print("=" * 40)
    print(f"TURN {turn}")
    print("MODEL CODE:")
    print(code)
    print("STDOUT:")
    print(stdout_log)
    print("VARIABLES:")
    print(f"{recompensa=}, {encerrado=}")
    print("=" * 40)

In [ ]:
# Reinicia o ambiente
obs, _ = env.reset()
encerrado = False
recompensa = 0.0
step_count = 0

# Cria um objeto para faciliar manter a "conversa" com o modelo
chat = GEMINI_CLIENT.chats.create(model=GEMINI_MODEL_ID, config={'system_instruction': SYSTEM_PROMPT_CODEACT, 'temperature': 0.3})

In [ ]:
# Contexto de execução para o "exec()", iniciado com variáveis iniciadas acima e com as funções de ação.
# Depois, ele será atualizado sucessivamente pelas chamadas a "exec()".
exec_context = {
    "obs": obs,
    "encerrado": encerrado,
    "recompensa": recompensa,
    "andar_para_frente": andar_para_frente,
    "girar_para_esquerda": girar_para_esquerda,
    "girar_para_direita": girar_para_direita,
    "abrir_porta": abrir_porta
}

In [ ]:
print("Running pure CodeAct agent...\n")

turn_num = 0

# Initial observation as the first user message
next_message = INITIAL_OBSERVATION_TEMPLATE.format(MISSION=obs["mission"],
                                                   GRID=obs["text_grid"])

print("INITIAL OBSERVATION:")
print(next_message)
print()

while not encerrado and turn_num < 20:
    # 1. Envia a próxima mensagem para o modelo
    model_response = chat.send_message(next_message)
    model_response_text = model_response.text

    # 2. Opcional - Imprime a resposta completa (para debugging apenas)
    #print(model_response_text)

    # 3. Verifica tags <execute>
    match = re.search(r"<execute>(.*?)</execute>", model_response_text, re.DOTALL)

    if not match:
        # If the model didn't return code, send an error message back to the model
        next_message = ("Sua resposta não seguiu o formato exigido.\n"
                        "Responda com código Python entre <execute> e </execute>.")
        continue

    code = match.group(1)

    # 4. Executa o código e captura stdout
    try:
        # obs.: o exec_context é passado por referência e vai ser alterado internamente
        stdout_output = run_codeact_snippet(code, {}, exec_context)
        turn_num += 1

    except Exception as e:
        # If execution fails, send an error message back to the model
        next_message = f"Execution error:\n{e}"
        continue

    # 5. Atualiza variáveis, lendo do "exec_context", que é alterado após a execução
    obs        = exec_context["obs"]
    encerrado  = exec_context["encerrado"]
    recompensa = exec_context["recompensa"]

    # 6. Imprime informações de debug (para nós, humanos)
    print_observation(turn_num, code, stdout_output, recompensa, encerrado)

    # 7. Prepara mensagem enviando stdout + estado atualizado como próxima observação
    next_message = NEXT_OBSERVATION_TEMPLATE.format(STDOUT_LOG=stdout_output)
    time.sleep(5.0)


# Informa o resultado final
if recompensa > 0:
    print("\nSUCESSO: O modelo chegou ao objetivo!")
else:
    print("\nFALHA: O agente morreu ou atingiu o limite de passos!")

env.close()